# phenoforge: end-to-end walkthrough

This notebook runs the **actual pipeline code** — the same node functions the LangGraph
agent calls, and the same engine functions the MCP server calls — step by step against
real data (`data/vocab.duckdb`, `data/phenotype_library/`, `data/concept_index.lance`).
Nothing here is a mock or a re-implementation; every call below is imported straight from
`phenoforge`.

**To try your own cohort description:** edit `POPULATION_DESCRIPTION` in the cell below and
re-run the notebook from Step 1.

Sections:
1. Setup (connect to the vocabulary DB, build retrievers)
2. Step-by-step pipeline: decompose → check curated → generate → confirm → assemble
3. The same thing over MCP (the transport a client like Claude Desktop would use)
4. The same thing as the real, compiled LangGraph agent (with a real pause-for-human interrupt)

## 0. Setup

Loads `.env` for `ANTHROPIC_API_KEY` (needed for Step 1's real Claude call), opens the real
vocabulary database, and builds the BM25 + dense retrievers shared by every step below —
exactly the objects `agent/graph.py` builds once and threads through every node.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Jupyter sets the kernel's cwd to this notebook's own directory, not the repo
# root, so relative data/ paths below would silently look in notebooks/data/.
# Walk up to the directory containing pyproject.toml and chdir there once
# (same pattern as notebooks/explore.ipynb).
_dir = Path.cwd()
while not (_dir / "pyproject.toml").exists():
    if _dir.parent == _dir:
        raise FileNotFoundError(f"Could not find repo root (pyproject.toml) above {Path.cwd()}")
    _dir = _dir.parent
os.chdir(_dir)
print(f"cwd: {Path.cwd()}")

load_dotenv()  # picks up ANTHROPIC_API_KEY from .env, same as scripts/run_agent.py

from phenoforge.engine.db import connect
from phenoforge.engine.retrieval import BM25Retriever
from phenoforge.engine.dense import DenseRetriever
from phenoforge.engine.models import ConceptSet

DB_PATH = Path("data/vocab.duckdb")
LIBRARY_DIR = Path("data/phenotype_library")
INDEX_PATH = Path("data/concept_index.lance")  # omit / set to None to run BM25-only

con = connect(DB_PATH)
bm25 = BM25Retriever(con)
dense = DenseRetriever(con, index_path=INDEX_PATH) if INDEX_PATH.exists() else None

print(f"BM25 ready. Dense index: {'loaded' if dense else 'not built — BM25-only fallback'}")

cwd: /Users/colbywilkinson/projects/phenoforge


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BM25 ready. Dense index: loaded


In [2]:
def concept_set_df(cs: ConceptSet) -> pd.DataFrame:
    """Render a ConceptSet as a readable table — for display only, not used by the pipeline."""
    rows = [
        {
            "code": c.concept_code,
            "name": c.concept_name,
            "tier": c.tier.value,
            "source": c.source,
        }
        for c in cs.concepts
    ]
    return pd.DataFrame(rows, columns=["code", "name", "tier", "source"])


def unmappable_df(cs: ConceptSet) -> pd.DataFrame:
    return pd.DataFrame([u.model_dump() for u in cs.unmappable], columns=["term", "reason"])

## Try it: the input you can swap

This is the only cell you need to edit to walk through a different cohort description.
Re-run every cell below it after changing this.

In [3]:
POPULATION_DESCRIPTION = "adults with iga nephropathy"

# A couple of others worth trying once you've run through the notebook once:
# POPULATION_DESCRIPTION = "patients with diabetic ketoacidosis"
# POPULATION_DESCRIPTION = "chronic kidney disease with sugar disease"  # tests semantic (dense) recall
# POPULATION_DESCRIPTION = "patients with a rare unrelated condition xyz"  # tests the unmappable path

POPULATION_DESCRIPTION

'adults with iga nephropathy'

## 1. The pipeline, one node at a time

These are the exact node functions in `phenoforge/agent/nodes.py` — the same functions
`agent/graph.py` wires into the compiled LangGraph. Here we call them directly, one at a
time, threading a `CohortAssemblyState` through by hand so each step's output is visible
before the next one runs. Section 4 runs the identical logic through the real compiled
graph, interrupt and all.

In [4]:
from phenoforge.agent import nodes
from phenoforge.agent.state import CohortAssemblyState

state = CohortAssemblyState(population_description=POPULATION_DESCRIPTION)
state

CohortAssemblyState(population_description='adults with iga nephropathy', seed_terms=[], term_results=[], pending_candidates=[], confirmed_codes=[], unmappable=[], final_concept_set=None)

### Step 1 — `decompose`

A real Claude call (`nodes.default_decomposer`) splits the free-text description into
distinct clinical seed terms, dropping demographic/study-design language ("adults", "with",
"and"). This is the piece named as *not yet built* back in `v0.5` (`phenoforge.eval`'s own
docstring) — the reason `v0.7` exists.

In [5]:
decompose_fn = nodes.default_decomposer()  # real Claude call — needs ANTHROPIC_API_KEY

update = nodes.decompose(state, decompose_fn=decompose_fn)
state = state.model_copy(update=update)

print(f"{POPULATION_DESCRIPTION!r}\n  -> {state.seed_terms}")

'adults with iga nephropathy'
  -> ['IgA nephropathy']


### Step 2 — `check_curated`

For each seed term, `engine/curated.find_curated_definition` runs hybrid (BM25 + dense)
search against the full ICD-10-CM vocabulary, then checks whether the top hit actually
belongs to one of the 8 bundled OHDSI Phenotype Library cohorts' *resolved code lists* —
matching on real code content, not the cohort's display name (see `DECISIONS.md` for why
that distinction mattered). A term only counts as `resolved` if this returns at least one
concept.

In [6]:
update = nodes.check_curated(state, con=con, library_dir=LIBRARY_DIR, bm25=bm25, dense=dense)
state = state.model_copy(update=update)

for result in state.term_results:
    print(f"\n=== {result.term!r} — resolved: {result.resolved} ===")
    if result.curated and result.curated.concepts:
        display(concept_set_df(result.curated))
    if result.curated and result.curated.unmappable:
        print("  (curated match reported unmappable items too:)")
        display(unmappable_df(result.curated))


=== 'IgA nephropathy' — resolved: False ===
  (curated match reported unmappable items too:)


,term,reason
0,IgA nephropathy,no bundled cohort matches this query


### Step 3 — `generate`

Only runs for terms that missed the curated library in Step 2. Every result here is
`generated`-tier by construction — hybrid search over the full ICD-10-CM vocabulary, no
peer-reviewed backing — and gets staged as a *pending candidate*, never folded straight into
the result. If every term resolved curated, this produces nothing and the whole next step is
skipped.

In [7]:
update = nodes.generate(state, bm25=bm25, dense=dense)
state = state.model_copy(update=update)

if state.pending_candidates:
    pending_df = pd.DataFrame(
        [
            {
                "#": i,
                "term": c.term,
                "code": c.concept.concept_code,
                "name": c.concept.concept_name,
                "source": c.concept.source,
            }
            for i, c in enumerate(state.pending_candidates)
        ]
    )
    display(pending_df)
else:
    print("Nothing generated — every seed term already resolved via the curated library.")

,#,term,code,name,source
0,0,IgA nephropathy,D80.2,Selective deficiency of immunoglobulin A [IgA],bm25:IgA nephropathy
1,1,IgA nephropathy,N02.B,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
2,2,IgA nephropathy,N15.0,Balkan nephropathy,bm25:IgA nephropathy
3,3,IgA nephropathy,N02.B9,Other recurrent and persistent immunoglobulin ...,dense:IgA nephropathy
4,4,IgA nephropathy,N14.0,Analgesic nephropathy,bm25:IgA nephropathy
5,5,IgA nephropathy,N02.B1,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
6,6,IgA nephropathy,N14.11,Contrast-induced nephropathy,bm25:IgA nephropathy
7,7,IgA nephropathy,N02.B6,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
8,8,IgA nephropathy,A36.84,Diphtheritic tubulo-interstitial nephropathy,bm25:IgA nephropathy
9,9,IgA nephropathy,N02.B5,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy


### Step 4 — `confirm` (the human-in-the-loop gate)

In the real graph (Section 4 below), this node calls `langgraph.types.interrupt(...)` and
the whole process **pauses** — no generated code reaches the final result without an
explicit human decision. Here, outside the graph runtime, we simulate that decision directly:
edit `ACCEPTED_INDICES` below to whichever row numbers from the table above you'd accept,
then re-run this cell.

In [8]:
ACCEPTED_INDICES: list[int] = list(range(len(state.pending_candidates)))  # accept everything shown above
# ACCEPTED_INDICES = [0]        # accept only the first candidate
# ACCEPTED_INDICES = []         # reject everything generated

accepted_codes = [state.pending_candidates[i].concept.concept_code for i in ACCEPTED_INDICES]
state = state.model_copy(update={"confirmed_codes": accepted_codes})
print(f"Accepted: {accepted_codes or '(none)'}")

Accepted: ['D80.2', 'N02.B', 'N15.0', 'N02.B9', 'N14.0', 'N02.B1', 'N14.11', 'N02.B6', 'A36.84', 'N02.B5']


### Step 5 — `assemble`

Curated concepts are included unconditionally. Generated candidates are included only if
their code made it into `confirmed_codes`; rejected ones turn into an explanatory
`unmappable` entry instead of silently vanishing. Deduplicated by concept id. The result is
never a bare code list — every entry keeps its tier and source.

In [9]:
update = nodes.assemble(state)
state = state.model_copy(update=update)

print("Final concept set:")
display(concept_set_df(state.final_concept_set))

if state.final_concept_set.unmappable:
    print("\nUnmappable / rejected:")
    display(unmappable_df(state.final_concept_set))

Final concept set:


,code,name,tier,source
0,D80.2,Selective deficiency of immunoglobulin A [IgA],generated,bm25:IgA nephropathy
1,N02.B,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
2,N15.0,Balkan nephropathy,generated,bm25:IgA nephropathy
3,N02.B9,Other recurrent and persistent immunoglobulin ...,generated,dense:IgA nephropathy
4,N14.0,Analgesic nephropathy,generated,bm25:IgA nephropathy
5,N02.B1,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
6,N14.11,Contrast-induced nephropathy,generated,bm25:IgA nephropathy
7,N02.B6,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
8,A36.84,Diphtheritic tubulo-interstitial nephropathy,generated,bm25:IgA nephropathy
9,N02.B5,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy



Unmappable / rejected:


,term,reason
0,IgA nephropathy,no bundled cohort matches this query


## 2. The same pipeline, over MCP

The MCP server (`phenoforge/mcp/server.py`) is a **thin transport layer** — no business
logic of its own (AGENTS.md). Every tool call below delegates to the exact same `engine`
functions used above; this is what a client like Claude Desktop would call over stdio. We
call the tool functions directly here (in-process) rather than spinning up a subprocess, so
you can see the call and the result together.

In [10]:
import phenoforge.mcp.server as mcp_server

mcp_server.configure(DB_PATH, library_dir=LIBRARY_DIR, index_path=INDEX_PATH)

# find_curated_definition: same engine.curated.find_curated_definition as Step 2 above
for term in state.seed_terms:
    result = mcp_server.find_curated_definition(term)
    print(f"\n=== find_curated_definition({term!r}) ===")
    if result.concepts:
        display(concept_set_df(result))
    else:
        display(unmappable_df(result))

[08/23/26 14:56:00] INFO     No device provided, using mps                                             ]8;id=4753804;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=4753805;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/sentence_transformers/base/model.py#190\190]8;;\

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753812;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4753813;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/FremyCompany/BioLORD-2023/resolve/main/modules.                
                             json "HTTP/1.1 307 Temporary Redirect"                                                

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753818;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4753819;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/FremyCompany/BioLORD-2                
                             023/167aab527b238a50ca65224e6319215d2ff4fc9f/modules.json "HTTP/1.1                   
                             200 OK"                                                                               

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753824;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4753825;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/FremyCompany/BioLORD-2023/resolve/main/config_s                
                             entence_transformers.json "HTTP/1.1 307 Temporary Redirect"                           

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753830;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4753831;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/FremyCompany/BioLORD-2                
                             023/167aab527b238a50ca65224e6319215d2ff4fc9f/config_sentence_transform                
                             ers.json "HTTP/1.1 200 OK"                                                            

...

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753993;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4753994;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/FremyCompany/BioLORD-2023/resolve/main/1_Poolin                
                             g/config.json "HTTP/1.1 307 Temporary Redirect"                                       

                    INFO     HTTP Request: HEAD                                                     ]8;id=4753999;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4754000;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/FremyCompany/BioLORD-2                
                             023/167aab527b238a50ca65224e6319215d2ff4fc9f/1_Pooling%2Fconfig.json                  
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: GET                                                      ]8;id=4754005;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4754006;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/FremyCompany/BioLORD-2023 "HTTP/1.1                 
                             200 OK"                                                                               


=== find_curated_definition('IgA nephropathy') ===


,term,reason
0,IgA nephropathy,no bundled cohort matches this query


In [11]:
# search_concepts: same engine.hybrid.hybrid_search as Step 3 above
result = mcp_server.search_concepts(POPULATION_DESCRIPTION, k=5)
print("search_concepts (generated-tier, hybrid BM25+dense):")
display(concept_set_df(result))

search_concepts (generated-tier, hybrid BM25+dense):


,code,name,tier,source
0,F50.83,Pica in adults,generated,bm25:adults with iga nephropathy
1,N02.B,Recurrent and persistent immunoglobulin A neph...,generated,dense:adults with iga nephropathy
2,D80.2,Selective deficiency of immunoglobulin A [IgA],generated,bm25:adults with iga nephropathy
3,N02.B9,Other recurrent and persistent immunoglobulin ...,generated,dense:adults with iga nephropathy
4,F50.84,Rumination disorder in adults,generated,bm25:adults with iga nephropathy


In [12]:
# expand_hierarchy: descendant expansion over the ICD-10-CM billing hierarchy
# (not exercised by the agent above, since it doesn't call this tool — shown here for completeness)
result = mcp_server.expand_hierarchy("E11")  # "Type 2 diabetes mellitus" and everything under it
print("expand_hierarchy('E11'):")
display(concept_set_df(result))

expand_hierarchy('E11'):


,code,name,tier,source
0,E11.1,Type 2 diabetes mellitus with ketoacidosis,generated,hierarchy_expansion:E11
1,E11.10,Type 2 diabetes mellitus with ketoacidosis wit...,generated,hierarchy_expansion:E11
2,E11.11,Type 2 diabetes mellitus with ketoacidosis wit...,generated,hierarchy_expansion:E11
3,E11.A,Type 2 diabetes mellitus without complications...,generated,hierarchy_expansion:E11
4,E11.0,Type 2 diabetes mellitus with hyperosmolarity,generated,hierarchy_expansion:E11
...,...,...,...,...
111,E11.29,Type 2 diabetes mellitus with other diabetic k...,generated,hierarchy_expansion:E11
112,E11.349,Type 2 diabetes mellitus with severe nonprolif...,generated,hierarchy_expansion:E11
113,E11.40,Type 2 diabetes mellitus with diabetic neuropa...,generated,hierarchy_expansion:E11
114,E11.49,Type 2 diabetes mellitus with other diabetic n...,generated,hierarchy_expansion:E11


## 3. The real, compiled LangGraph agent

Everything above ran the pipeline manually so each step was visible in isolation. This runs
the actual compiled graph (`agent/graph.build_graph`) — same topology as the diagram in
`agent/graph.py`'s docstring:

```
decompose -> check_curated -> [generate -> confirm] -> assemble
```

with the bracketed segment skipped automatically when every term resolves curated. The
graph genuinely **pauses execution** at `confirm` via `langgraph.types.interrupt` — the cell
below will show `__interrupt__` in the result if there's anything generated to review, and
the following cell resumes it with `Command(resume=...)`, exactly like `scripts/run_agent.py`
does at the command line.

In [13]:
from langgraph.types import Command

from phenoforge.agent.graph import build_graph

graph = build_graph(con, LIBRARY_DIR, dense=dense)
config = {"configurable": {"thread_id": "walkthrough-notebook"}}

result = graph.invoke(
    {"population_description": POPULATION_DESCRIPTION}, config=config
)

if "__interrupt__" in result:
    pending = result["__interrupt__"][0].value["pending"]
    print(f"Graph paused — {len(pending)} generated candidate(s) awaiting confirmation:")
    display(pd.DataFrame(pending))
else:
    print("Graph ran straight through — every term resolved curated, no pause needed.")
    display(concept_set_df(result["final_concept_set"]))

[08/23/26 14:56:38] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=4754011;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=4754012;file:///Users/colbywilkinson/projects/phenoforge/.venv/lib/python3.14/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

Graph paused — 10 generated candidate(s) awaiting confirmation:


,term,concept_code,concept_name,source
0,IgA nephropathy,D80.2,Selective deficiency of immunoglobulin A [IgA],bm25:IgA nephropathy
1,IgA nephropathy,N02.B,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
2,IgA nephropathy,N15.0,Balkan nephropathy,bm25:IgA nephropathy
3,IgA nephropathy,N02.B9,Other recurrent and persistent immunoglobulin ...,dense:IgA nephropathy
4,IgA nephropathy,N14.0,Analgesic nephropathy,bm25:IgA nephropathy
5,IgA nephropathy,N02.B1,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
6,IgA nephropathy,N14.11,Contrast-induced nephropathy,bm25:IgA nephropathy
7,IgA nephropathy,N02.B6,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy
8,IgA nephropathy,A36.84,Diphtheritic tubulo-interstitial nephropathy,bm25:IgA nephropathy
9,IgA nephropathy,N02.B5,Recurrent and persistent immunoglobulin A neph...,dense:IgA nephropathy


**If the graph paused above**, resume it here with the codes to accept (edit
`RESUME_CODES` to whichever `concept_code`s from the pending table you want to include, or
leave it as `"all"` to accept everything). If it *didn't* pause, skip this cell — the graph
already finished above.

In [14]:
RESUME_CODES: list[str] | str = "all"  # or e.g. ["E11.21", "E11.9"]

if "__interrupt__" in result:
    pending = result["__interrupt__"][0].value["pending"]
    codes = [p["concept_code"] for p in pending] if RESUME_CODES == "all" else RESUME_CODES
    result = graph.invoke(Command(resume=codes), config=config)
    print(f"Resumed with: {codes}\n")

final = result["final_concept_set"]
print("Agent's final concept set:")
display(concept_set_df(final))
if final.unmappable:
    print("\nUnmappable / rejected:")
    display(unmappable_df(final))

Resumed with: ['D80.2', 'N02.B', 'N15.0', 'N02.B9', 'N14.0', 'N02.B1', 'N14.11', 'N02.B6', 'A36.84', 'N02.B5']

Agent's final concept set:


,code,name,tier,source
0,D80.2,Selective deficiency of immunoglobulin A [IgA],generated,bm25:IgA nephropathy
1,N02.B,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
2,N15.0,Balkan nephropathy,generated,bm25:IgA nephropathy
3,N02.B9,Other recurrent and persistent immunoglobulin ...,generated,dense:IgA nephropathy
4,N14.0,Analgesic nephropathy,generated,bm25:IgA nephropathy
5,N02.B1,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
6,N14.11,Contrast-induced nephropathy,generated,bm25:IgA nephropathy
7,N02.B6,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy
8,A36.84,Diphtheritic tubulo-interstitial nephropathy,generated,bm25:IgA nephropathy
9,N02.B5,Recurrent and persistent immunoglobulin A neph...,generated,dense:IgA nephropathy



Unmappable / rejected:


,term,reason
0,IgA nephropathy,no bundled cohort matches this query


---

**To explore further:** change `POPULATION_DESCRIPTION` near the top and re-run from
Step 1 (or just re-run Section 3, the compiled graph, on its own — it doesn't depend on the
manual walkthrough state). A few descriptions worth trying, given the 8 bundled cohorts
(type 1/2/gestational diabetes, DKA, retinopathy, chronic kidney disease):

- `"adults with type 2 diabetes and diabetic nephropathy"` — the README's own example
- `"patients with diabetic ketoacidosis"` — should resolve curated cleanly and directly
- `"chronic kidney disease with sugar disease"` — tests semantic (dense) recall on a
  paraphrase that shares no vocabulary with "diabetes"
- `"patients with a rare unrelated condition xyz"` — tests the unmappable path end to end